# Goods Trade Data Cleaning

Use only CSO TSA14 for 2023-2025. Prepare partner rankings, product rankings and three-year comparisons.

1. Set up files.
2. Check the data.
3. Clean the data.
4. Save and check Tableau files.

Source: https://data.cso.ie/table/TSA14

## Step 1: Manage the Environment

Read the saved JSON with Python and pandas. Keep the original file unchanged.

In [1]:
import json
import hashlib
import math
from pathlib import Path
from itertools import product
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value.to_string() if hasattr(value, 'to_string') else value)

pd.set_option('display.max_columns', 14)
pd.set_option('display.precision', 4)
data_folder = next((folder for base in [Path.cwd(), *Path.cwd().parents]
                         for folder in [base, base / 'data',
                                        base / 'Visualisation implementation' / 'data']
                         if (folder / 'TSA14.json').exists()), None)
if data_folder is None:
    raise FileNotFoundError('Run from this repository or its data folder containing TSA14.json.')
source_file = data_folder / 'TSA14.json'
output_folder = data_folder
source_bytes = source_file.read_bytes()
source = json.loads(source_bytes)
years = [2023, 2024, 2025]
print('Source:', source['label'])
print('Updated:', source.get('updated'))

Source: Value of Merchandise Trade
Updated: 2026-08-18T11:00:00.000Z


## Step 2: Understand the Data

### 2.1 Load the data

Each row is one year, flow, partner and product. Read codes in the order given by JSON. Keep codes as text.

In [2]:
def ordered_codes(category):
    index = category['index']
    return index if isinstance(index, list) else sorted(index, key=index.get)

ids = source['id']
assert [source['dimension'][key]['label'] for key in ids] == [
    'Statistic', 'Year', 'Countries and Territories', 'Commodity Group']
axes = [ordered_codes(source['dimension'][key]['category']) for key in ids]
assert [len(a) for a in axes] == source['size']
keys = list(product(*axes))
values = source['value']
if isinstance(values, dict):
    values = [values.get(str(i)) for i in range(len(keys))]
assert len(values) == len(keys)
df_raw = pd.DataFrame(keys, columns=['statistic_code', 'year_code', 'partner_code', 'product_code'])
df_raw['value_eur_thousand'] = values
for position, column, name in [(0, 'statistic_code', 'statistic'), (2, 'partner_code', 'partner'),
                                (3, 'product_code', 'product')]:
    labels = source['dimension'][ids[position]]['category']['label']
    df_raw[name] = df_raw[column].map(labels)
print('Rows:', len(df_raw))
display(df_raw.head())
df_raw.info()

Rows: 95940
  statistic_code year_code partner_code product_code  value_eur_thousand         statistic  partner                                                    product
0       TSA14C01      2023           AD       001001                 NaN  Value of Imports  Andorra                         Live animals except fish etc. (00)
1       TSA14C01      2023           AD       001002                 NaN  Value of Imports  Andorra                            Meat and meat preparations (01)
2       TSA14C01      2023           AD       001003                 NaN  Value of Imports  Andorra                         Dairy products and birds eggs (02)
3       TSA14C01      2023           AD       001004                 NaN  Value of Imports  Andorra  Fish, crustaceans, molluscs and preparations thereof (03)
4       TSA14C01      2023           AD       001005              0.0384  Value of Imports  Andorra                       Cereals and cereal preparations (04)
<class 'pandas.core.frame.DataFram

### 2.2 Summarise and check

Check years, amounts, duplicate rows and missing values. Both flows use thousand euro.

In [3]:
units = source['dimension'][ids[0]]['category']['unit']
assert all(unit['label'] == 'Euro Thousand' for unit in units.values())
quality_report = pd.DataFrame({
    'data_type': df_raw.dtypes.astype(str),
    'missing_values': df_raw.isna().sum(),
    'missing_pct': df_raw.isna().mean().mul(100).round(2),
    'unique_values': df_raw.nunique(),
})
display(quality_report)
display(df_raw.groupby('statistic')['value_eur_thousand'].describe())
display(df_raw.select_dtypes(include=['object']).describe().T)
print('Exact duplicate rows:', df_raw.duplicated().sum())
print('Genuine zero amounts:', df_raw.value_eur_thousand.eq(0).sum())
display(df_raw.groupby(['year_code', 'statistic']).agg(
    rows=('value_eur_thousand', 'size'), reported=('value_eur_thousand', 'count'),
    missing=('value_eur_thousand', lambda x: x.isna().sum())))

                   data_type  missing_values  missing_pct  unique_values
statistic_code        object               0         0.00              2
year_code             object               0         0.00              3
partner_code          object               0         0.00            246
product_code          object               0         0.00             65
value_eur_thousand   float64           54499        56.81          34800
statistic             object               0         0.00              2
partner               object               0         0.00            246
product               object               0         0.00             65
                    count        mean          std    min     25%       50%        75%         max
statistic                                                                                         
Value of Exports  19075.0  35578.7124  772698.8531  0.001  7.0415  130.7510  1804.7154  8.1910e+07
Value of Imports  22366.0  18764.5661  195208.

## Step 3: Clean the Data

### 3.1 Tidy fields and units

Use integer years and text codes. Convert thousand euro to billion euro by dividing by 1,000,000. Do not round the saved values.

In [4]:
df = df_raw.copy()
for column in ['statistic_code', 'partner_code', 'product_code', 'statistic', 'partner', 'product']:
    df[column] = df[column].astype('string').str.strip()
    assert df[column].notna().all() and df[column].str.len().gt(0).all()
df['year'] = pd.to_numeric(df.pop('year_code'), errors='raise').astype('int64')
df = df.loc[df.year.isin(years)].copy()
assert set(df.year) == set(years)
df['trade_flow'] = df.statistic.map({'Value of Imports': 'Imports', 'Value of Exports': 'Exports'}).astype('string')
assert df.trade_flow.notna().all()
df['value_eur_thousand'] = pd.to_numeric(df.value_eur_thousand, errors='raise')
df['value_eur_billion'] = df.value_eur_thousand / 1_000_000
# Separate missing values from real zeros.
df['reported_count'] = df.value_eur_thousand.notna().astype('int64')
df['missing_count'] = df.value_eur_thousand.isna().astype('int64')
df['record_count'] = 1
df['observation_status'] = df.value_eur_thousand.notna().map({True: 'reported', False: 'unavailable'})
# Q codes are special partner categories. IE is Ireland itself.
df['is_named_partner'] = ~df.partner_code.str.startswith('Q') & df.partner_code.ne('IE')
df['is_named_product'] = df.product_code.ne('012')
df['partner_product_key'] = df.partner_code + ':' + df.product_code

### 3.2 Keep blanks and check keys

Blank means no value is provided. Do not fill it with zero. The reason for each blank is not given.
CSO can hide confidential trade details: [section 9.2](https://www.cso.ie/en/media/csoie/methods/goodsexportsandimports/Goods_Exports_and_Imports__SIMS2024.pdf).

Keep special categories in the data but exclude them from named Top 10 rankings. Keep Great Britain and Northern Ireland separate.

In [5]:
before = len(df)
df = df.drop_duplicates().copy()
print('Exact duplicates removed:', before - len(df))
key_columns = ['year', 'trade_flow', 'partner_code', 'product_code']
assert not df.duplicated(key_columns).any(), 'Conflicting trade values.'
assert len(df) == len(df_raw), 'Review any row removal.'
assert df.value_eur_thousand.dropna().ge(0).all()
assert df.value_eur_thousand.dropna().map(math.isfinite).all()
assert df.value_eur_thousand.isna().sum() == df_raw.value_eur_thousand.isna().sum()
assert df.value_eur_thousand.eq(0).sum() == df_raw.value_eur_thousand.eq(0).sum()
assert df.value_eur_billion.isna().equals(df.value_eur_thousand.isna())
assert (df.reported_count + df.missing_count).eq(df.record_count).all()
df = df.sort_values(key_columns).reset_index(drop=True)
display(df.loc[~df.is_named_partner, ['partner_code', 'partner']].drop_duplicates())
display(df.head())

Exact duplicates removed: 0
      partner_code                           partner
6240            IE                           Ireland
11310           QA                             Qatar
11375           QP                         High seas
11440           QQ     Stores and Provisions Unknown
11505           QR  Stores and Provisions (Intra EU)
11570           QS    Stores and Provisions (Non EU)
11635           QV                Unknown (Intra EU)
11700           QW                  Unknown (Non-EU)
  statistic_code partner_code product_code  value_eur_thousand         statistic  partner                                                    product  year trade_flow  value_eur_billion  reported_count  missing_count  record_count observation_status  is_named_partner  is_named_product partner_product_key
0       TSA14C02           AD       001001                 NaN  Value of Exports  Andorra                         Live animals except fish etc. (00)  2023    Exports                NaN      

## Step 4: Prepare Tableau Data

### 4.1 Check sums and missing counts

Sum available values only. An all-blank group stays blank. Keep missing counts beside each sum.
These are available-value sums, not verified national totals. Missing records can affect rankings and year comparisons.

In [6]:
def summarise(table, group_columns):
    grouped = table.groupby(group_columns, dropna=False)
    result = grouped.agg(record_count=('record_count', 'sum'),
                         reported_count=('reported_count', 'sum'),
                         missing_count=('missing_count', 'sum')).reset_index()
    amounts = grouped['value_eur_billion'].sum(min_count=1).rename('available_value_eur_billion').reset_index()
    result = result.merge(amounts, on=group_columns, validate='one_to_one')
    result['all_values_reported'] = result.missing_count.eq(0)
    result['reported_cell_pct'] = 100 * result.reported_count / result.record_count
    assert result.loc[result.reported_count.eq(0), 'available_value_eur_billion'].isna().all()
    return result

coverage = summarise(df, ['year', 'trade_flow'])
partner_check = summarise(df, ['year', 'trade_flow', 'partner_code', 'partner'])
product_check = summarise(df, ['year', 'trade_flow', 'product_code', 'product'])
assert partner_check.record_count.eq(len(axes[3])).all()
assert product_check.record_count.eq(len(axes[2])).all()
display(coverage)
# Keep all partners and products. Tableau will calculate Top 10 after filters.
display(partner_check.loc[(partner_check.year == 2025) & partner_check.trade_flow.eq('Exports')]
        .sort_values('available_value_eur_billion', ascending=False).head(10))

   year trade_flow  record_count  reported_count  missing_count  available_value_eur_billion  all_values_reported  reported_cell_pct
0  2023    Exports         15990            6387           9603                     196.0297                False            39.9437
1  2023    Imports         15990            7506           8484                     140.2945                False            46.9418
2  2024    Exports         15990            6372           9618                     223.7340                False            39.8499
3  2024    Imports         15990            7442           8548                     134.6556                False            46.5416
4  2025    Exports         15990            6316           9674                     258.9002                False            39.4997
5  2025    Imports         15990            7418           8572                     144.7382                False            46.3915
      year trade_flow partner_code           partner  record_count  r

### 4.2 Save and reload

Use `goods_trade_dashboard.csv` as the Tableau source. The coverage CSV is a check table, not extra trade amounts.
Keep every year and category so selections can update all charts. Import codes as text. CSV blanks stay missing.

In [7]:
main_file = output_folder / 'goods_trade_dashboard.csv'
df.to_csv(main_file, index=False, encoding='utf-8-sig', na_rep='')
coverage.to_csv(output_folder / 'trade_coverage.csv', index=False, encoding='utf-8-sig', na_rep='')
reloaded = pd.read_csv(main_file, keep_default_na=False,
    na_values={'value_eur_thousand': [''], 'value_eur_billion': ['']},
    dtype={column: 'string' for column in ['statistic_code', 'partner_code', 'product_code', 'statistic',
                                          'partner', 'product', 'trade_flow', 'observation_status', 'partner_product_key']})
assert len(reloaded) == len(df)
assert not reloaded.duplicated(key_columns).any()
for column in ['partner_code', 'product_code', 'partner_product_key']:
    assert reloaded[column].equals(df[column])
for column in ['value_eur_thousand', 'value_eur_billion']:
    pd.testing.assert_series_equal(reloaded[column], df[column], check_exact=False, rtol=1e-12, atol=1e-12)
assert reloaded.value_eur_thousand.isna().sum() == df.value_eur_thousand.isna().sum()
assert reloaded.value_eur_thousand.eq(0).sum() == df.value_eur_thousand.eq(0).sum()
# Check original JSON positions, including interior records.
for s, y, p, c in [(0, 0, 0, 0), (1, 1, 100, 31), (1, 2, len(axes[2])-1, len(axes[3])-1)]:
    offset = ((s * len(axes[1]) + y) * len(axes[2]) + p) * len(axes[3]) + c
    match = df.loc[df.statistic_code.eq(axes[0][s]) & df.year.eq(int(axes[1][y]))
                   & df.partner_code.eq(axes[2][p]) & df.product_code.eq(axes[3][c]), 'value_eur_thousand']
    assert len(match) == 1
    expected, actual = values[offset], match.iloc[0]
    assert pd.isna(actual) if expected is None else actual == expected
assert source_file.read_bytes() == source_bytes
manifest = {
    'source_table': 'TSA14', 'table_url': 'https://data.cso.ie/table/TSA14',
    'api_url': 'https://ws.cso.ie/public/api.restful/PxStat.Data.Cube_API.ReadDataset/TSA14/JSON-stat/2.0/en',
    'source_updated': source.get('updated'), 'sha256': hashlib.sha256(source_bytes).hexdigest(),
    'years': years, 'rows': len(df), 'partners': int(df.partner_code.nunique()),
    'products': int(df.product_code.nunique()), 'reported_amounts': int(df.reported_count.sum()),
    'missing_amounts': int(df.missing_count.sum()), 'genuine_zeros': int(df.value_eur_thousand.eq(0).sum()),
    'unit_conversion': 'Euro Thousand / 1,000,000 = Euro Billion',
    'missing_policy': 'Keep nulls. Exact missing reason unknown. All-missing selections stay blank.',
    'sum_label': 'Sum of available values, not verified official national totals.',
    'ranking_policy': 'Named partners/products only. Calculate Top 10 after year and flow selections.',
    'outputs': ['goods_trade_dashboard.csv', 'trade_coverage.csv', 'processing_manifest.json'],
}
(output_folder / 'processing_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('Saved:', main_file)
print('Rows:', len(df), '| reported:', manifest['reported_amounts'], '| missing:', manifest['missing_amounts'])
print('All processing and export checks passed.')

Saved: /Users/xiaoli/Desktop/ucd26autumn/Information visulization/final assignment/data/trade/processed/tsa14/goods_trade_dashboard.csv
Rows: 95940 | reported: 41441 | missing: 54499
All processing and export checks passed.


### Tableau setup

- Default: 2025 and Exports.
- Top: three-year Imports/Exports bars. Keep all three years.
- Middle: Top 10 named partners for the selected year and flow.
- Bottom: Top 10 named products. A partner selection filters this chart.
- Partner and product selections filter the top chart together. Update its title.
- The year control changes rankings, not the top chart's year range.
- Reset clears the partner and product selections.
- Use `SUM(reported_count)` and `SUM(missing_count)` in tooltips.
- If reported count is zero, show “No value provided”. Never turn blanks into zero.
- Label amounts “Sum of available values”. Missingness may differ between years.

### Short summary for the report

We used TSA14 for 2023-2025 only. We checked codes and duplicate keys, converted amounts to billion euro, and kept all missing values. Special partner categories and unclassified estimates remain in the data but are excluded from named rankings. All charts use the same detail table. Sums include available values only, with missing counts shown separately.